In [ ]:
# ═══════════════════════════════════════════════════════════════
# Charm Bulk Workflow — One-Click Run
# ═══════════════════════════════════════════════════════════════
# Edit CONFIG_PATH below to point to your flow config YAML.
# Then run this cell to execute the full workflow.
# ═══════════════════════════════════════════════════════════════

import yaml
import os
import sys
import subprocess
import time
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────
# Edit this path to your workflow config file
CONFIG_PATH = "config_flow.yml"

# Options
WORKERS = 4           # Number of parallel workers
CORRELATED = True     # True for correlated cuts, False for combined
PYTHON = "conda run -n alice python3"  # Python interpreter (with ROOT + flarefly)

# Workflow root directory (auto-detected from this notebook's location)
WF_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
SRC_DIR = os.path.join(WF_DIR, "src")


def run_cmd(cmd, label=""):
    """Run a shell command with real-time output."""
    print(f"[{label}] Running: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=False, timeout=600)
    if result.returncode != 0:
        print(f"[{label}] COMMAND FAILED (rc={result.returncode})")
        raise RuntimeError(f"Command failed: {cmd}")
    return result


def main():
    t0 = time.time()
    
    if not os.path.exists(CONFIG_PATH):
        print(f"[ERROR] Config file not found: {CONFIG_PATH}")
        return
    
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)
    
    operations = config.get("operations", {})
    base_outdir = config.get("outdir", ".")
    suffix = config.get("suffix", "default")
    
    # Determine output directory
    corr_str = "_correlated" if CORRELATED else "_combined"
    if "_correlated" not in base_outdir and "_combined" not in base_outdir:
        outdir = os.path.join(base_outdir, f"cutvar_{suffix}{corr_str}")
    else:
        outdir = base_outdir
    
    os.makedirs(outdir, exist_ok=True)
    
    # Copy config to output
    cfg_copy_dir = os.path.join(outdir, "config_flow")
    os.makedirs(cfg_copy_dir, exist_ok=True)
    os.system(f"cp {CONFIG_PATH} {cfg_copy_dir}/")
    
    corr_flag = "--correlated -c" if CORRELATED else ""
    
    # ── Step 1: make_cutsets_cfgs ──────────────────────────────
    if operations.get("make_yaml", False):
        print("\n" + "═" * 60)
        print("  Step 1/4: Generating cutset YAML files")
        print("═" * 60)
        cmd = f"cd {WF_DIR} && {PYTHON} {SRC_DIR}/make_cutsets_cfgs.py {CONFIG_PATH} -o {outdir} {corr_flag}"
        run_cmd(cmd, "CUTSETS")
        
        cutsets_dir = os.path.join(outdir, "cutsets")
        n_sets = len([f for f in os.listdir(cutsets_dir) if f.endswith(".yml")])
        print(f"  → Created {n_sets} cutset files")
    else:
        print("\n  Step 1/4: make_yaml — SKIPPED")
    
    # ── Step 2: mass_fit ───────────────────────────────────────
    if operations.get("mass_fit", False):
        print("\n" + "═" * 60)
        print("  Step 2/4: Invariant mass fitting (flarefly)")
        print("═" * 60)
        
        proj_dir = config.get("inputs", {}).get("projs", "") or os.path.join(outdir, "projs")
        ry_dir = os.path.join(outdir, "raw_yields")
        os.makedirs(ry_dir, exist_ok=True)
        
        proj_files = sorted(f for f in os.listdir(proj_dir) if f.startswith("proj_") and f.endswith(".root"))
        print(f"  Found {len(proj_files)} projection files in {proj_dir}")
        
        for pf in proj_files:
            proj_path = os.path.join(proj_dir, pf)
            cmd = f"CUDA_VISIBLE_DEVICES=-1 {PYTHON} {SRC_DIR}/mass_fit.py {CONFIG_PATH} {proj_path} -o {ry_dir} -b"
            run_cmd(cmd, "MASSFIT")
    else:
        print("\n  Step 2/4: mass_fit — SKIPPED")
    
    # ── Step 3: cut_variation ──────────────────────────────────
    if operations.get("cut_variation", False):
        print("\n" + "═" * 60)
        print("  Step 3/4: Cut variation")
        print("═" * 60)
        
        ry_dir = config.get("inputs", {}).get("raw_yields", "") or os.path.join(outdir, "raw_yields")
        eff_dir = config.get("inputs", {}).get("effs", "") or os.path.join(outdir, "effs")
        
        cmd = f"cd {WF_DIR} && CUDA_VISIBLE_DEVICES=-1 {PYTHON} {SRC_DIR}/cut_variation.py {CONFIG_PATH} {ry_dir} {eff_dir} -b"
        run_cmd(cmd, "CUTVAR")
    else:
        print("\n  Step 3/4: cut_variation — SKIPPED")
    
    # ── Step 4: data_driven_fraction ───────────────────────────
    if operations.get("data_driven_fraction", False):
        print("\n" + "═" * 60)
        print("  Step 4/4: Data-driven fraction")
        print("═" * 60)
        
        eff_dir = config.get("inputs", {}).get("effs", "") or os.path.join(outdir, "effs")
        cutvar_path = os.path.join(outdir, "cutVar", "cutVar.root")
        frac_dir = os.path.join(outdir, "frac")
        
        cmd = f"cd {WF_DIR} && CUDA_VISIBLE_DEVICES=-1 {PYTHON} {SRC_DIR}/data_driven_fraction.py {cutvar_path} {eff_dir} -o {frac_dir} -b"
        run_cmd(cmd, "FRACTION")
    else:
        print("\n  Step 4/4: data_driven_fraction — SKIPPED")
    
    elapsed = time.time() - t0
    print("\n" + "═" * 60)
    print(f"  Workflow completed in {elapsed:.1f} s")
    print(f"  Output: {outdir}")
    print("═" * 60)


if __name__ == "__main__":
    main()
